<a href="https://colab.research.google.com/github/oscarabrilarranz/Pruebas/blob/main/Caidasmas5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf
import pandas as pd
import requests
from io import StringIO

UMBRAL_CAIDA = -5.0


def leer_tabla_wikipedia(url):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/125.0 Safari/537.36"
        )
    }

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    return pd.read_html(StringIO(response.text))


def obtener_sp500():
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    tabla = leer_tabla_wikipedia(url)[0]

    tickers = tabla["Symbol"].tolist()
    nombres = tabla["Security"].tolist()

    tickers = [str(t).replace(".", "-") for t in tickers]

    empresas = dict(zip(tickers, nombres))

    print(f"S&P500 cargado: {len(tickers)} acciones")

    return tickers, empresas


def obtener_ibex35():
    url = "https://en.wikipedia.org/wiki/IBEX_35"
    tablas = leer_tabla_wikipedia(url)

    tickers = []
    nombres = []

    for tabla in tablas:
        cols = [str(c) for c in tabla.columns]

        if "Ticker" in cols:
            tickers = tabla["Ticker"].tolist()

            if "Company" in cols:
                nombres = tabla["Company"].tolist()
            elif "Name" in cols:
                nombres = tabla["Name"].tolist()
            else:
                nombres = tickers

            break

    if len(tickers) == 0:
        raise Exception("No se encontró la tabla del IBEX 35")

    tickers = [
        str(t) if str(t).endswith(".MC") else str(t) + ".MC"
        for t in tickers
    ]

    empresas = dict(zip(tickers, nombres))

    print(f"IBEX35 cargado: {len(tickers)} acciones")

    return tickers, empresas


def analizar_caidas(tickers, empresas, mercado):
    print(f"\nDescargando datos de {mercado}...")

    datos = yf.download(
        tickers,
        period="2d",
        interval="1d",
        auto_adjust=True,
        group_by="ticker",
        progress=False,
        threads=True
    )

    resultados = []

    for ticker in tickers:
        try:
            cierre_hoy = datos[ticker]["Close"].iloc[-1]
            cierre_ayer = datos[ticker]["Close"].iloc[-2]

            variacion = ((cierre_hoy - cierre_ayer) / cierre_ayer) * 100

            resultados.append({
                "Mercado": mercado,
                "Empresa": empresas.get(ticker, ""),
                "Ticker": ticker,
                "Cierre ayer": round(cierre_ayer, 2),
                "Cierre hoy": round(cierre_hoy, 2),
                "Variación %": round(variacion, 2)
            })

        except Exception:
            continue

    df = pd.DataFrame(resultados)

    if df.empty:
        return df

    df = df.sort_values(by="Variación %")

    return df[df["Variación %"] <= UMBRAL_CAIDA]


def main():
    sp500, empresas_sp500 = obtener_sp500()
    ibex35, empresas_ibex35 = obtener_ibex35()

    caidas_sp500 = analizar_caidas(sp500, empresas_sp500, "S&P500")
    caidas_ibex35 = analizar_caidas(ibex35, empresas_ibex35, "IBEX35")

    resultado = pd.concat(
        [caidas_sp500, caidas_ibex35],
        ignore_index=True
    )

    print("\n========================")
    print("ACCIONES CON CAÍDAS > 5%")
    print("========================\n")

    if resultado.empty:
        print("No se encontraron valores con caídas superiores al 5%.")
    else:
        print(resultado.to_string(index=False))

        resultado.to_csv(
            "acciones_caidas_mas_5pct.csv",
            index=False
        )

        print("\nCSV generado: acciones_caidas_mas_5pct.csv")


if __name__ == "__main__":
    main()

S&P500 cargado: 503 acciones
IBEX35 cargado: 35 acciones

Descargando datos de S&P500...

Descargando datos de IBEX35...

ACCIONES CON CAÍDAS > 5%

Mercado                    Empresa Ticker  Cierre ayer  Cierre hoy  Variación %
 S&P500               Corning Inc.    GLW       208.28      191.81        -7.91
 S&P500                   Coinbase   COIN       212.01      195.43        -7.82
 S&P500         Ford Motor Company      F        14.48       13.40        -7.46
 S&P500          Micron Technology     MU       776.01      724.66        -6.62
 S&P500                      Ciena   CIEN       591.57      554.46        -6.27
 S&P500                    Newmont    NEM       116.33      109.06        -6.25
 S&P500                      Intel   INTC       115.93      108.77        -6.18
 S&P500                 Supermicro   SMCI        33.03       31.04        -6.02
 S&P500     Advanced Micro Devices    AMD       449.70      424.10        -5.69
 S&P500      Albemarle Corporation    ALB       191.